# 细胞 Aβ / Iba1 / CD68 批量分析

这个 Notebook 用于 细胞免疫荧光：分割 Aβ 斑块，测量整个分析区域及斑块周围距离环中的 Iba1、CD68 和二维重叠。支持从不同文件夹反复添加显微镜图像、逐文件记忆参数，并把当前整套参数一键应用到所有文件。

In [1]:
from copy import deepcopy
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# 项目路径。若整个项目被移动，只需修改这一行。
PROJECT_ROOT = Path(r"E:\projects\cell_analyzer")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from brain_section_analyzer.config import DEFAULT_CONFIG
from brain_section_analyzer.interactive import launch_brain_section_tuning_widget


## 1. 设置输出位置和可选的初始文件

`IMAGE_PATHS` 可以保持为空，之后直接点击界面的 **Add images**。这个按钮可以重复使用，因此能够从不同文件夹添加文件。正式定量建议 `Final zoom = 1.0`；调参时可使用较小的 `Preview zoom`。

In [2]:
OUTPUT_ROOT = Path(r"I:\QIULAB\data\AD_PAC\cell_analysis_results")
IMAGE_PATHS = []  # 可选：在这里填写绝对路径，也可以在面板中添加

config = deepcopy(DEFAULT_CONFIG)

# 以下参数只是启动值，均可在面板中调整。
config["input"]["z_projection"] = "max"
config["input"]["zoom"] = 1.0

# 使用第 4 个 DAPI 通道分割细胞核，再用核周 Iba1 阳性确认小胶质细胞。
# 计数范围直接复用下方 ring_edges_um；每个细胞按核中心分配给最近 plaque。
config["channels"]["dapi"]["enabled"] = True
config["microglia_count"].update({
    "enabled": True,
    "min_nucleus_area_um2": 10.0,
    "max_nucleus_area_um2": 150.0,
    "min_circularity": 0.20,
    "min_solidity": 0.70,
    "max_eccentricity": 0.98,
    "perinuclear_radius_um": 3.0,
    "min_iba1_positive_fraction": 0.15,
})

# Iba1/CD68：先做强度阈值，再按连通对象的面积和形态过滤。
# 下列默认值等同于不过滤；请在面板预览代表图后再设定，并对同批 WT/PAC-cKO 使用同一套参数。
for marker in ("iba1", "cd68"):
    config["channels"][marker]["object_filter"].update({
        "enabled": True,
        "opening_radius_px": 0,
        "closing_radius_px": 0,
        "fill_holes": False,
        "max_hole_area_um2": 0.0,
        "min_area_um2": 0.0,
        "max_area_um2": None,
        "min_circularity": 0.0,
        "min_solidity": 0.0,
        "max_eccentricity": 1.0,
    })
# Aβ 斑块：面积、形态学、孔洞过滤和接触斑块分割。
config["plaque"]["min_area_um2"] = 10.0
config["plaque"]["max_area_um2"] = None
config["plaque"]["opening_radius_px"] = 0
config["plaque"]["closing_radius_px"] = 1
config["plaque"]["max_hole_area_um2"] = 0.0
config["plaque"]["min_circularity"] = 0.0
config["plaque"]["min_solidity"] = 0.0
config["plaque"]["max_eccentricity"] = 1.0

# 排除神经元样 Aβ：先用较低阈值连出胞体，再结合外轮廓、尺寸和暗核判断。
# Soma exclusion uses Aβ only; the optional final gate below uses DAPI/Iba1. CD68 is never used.
config["plaque"]["neuron_exclusion_mode"] = "shape_and_dark_center"
config["plaque"]["neuron_detection_threshold_scale"] = 0.75
config["plaque"]["neuron_min_diameter_um"] = 8.0
config["plaque"]["neuron_max_diameter_um"] = 28.0
config["plaque"]["neuron_min_circularity"] = 0.35
config["plaque"]["neuron_min_solidity"] = 0.60
config["plaque"]["neuron_min_hole_fraction"] = 0.03
config["plaque"]["neuron_max_center_shell_ratio"] = 0.95

# Final plaque gate: distance is measured outward from each plaque edge.
config["plaque"]["require_nearby_microglia"] = True
config["plaque"]["nearby_microglia_radius_um"] = 30.0
config["plaque"]["min_nearby_microglia_count"] = 1
config["plaque"]["split_touching"] = False
config["plaque"]["min_peak_distance_px"] = 8

# 所有半径均为累计区域并包含 plaque 本体；[0, 15, 30] 生成 plaque+外侧0–15 µm 和 plaque+外侧0–30 µm。
config["spatial"]["ring_edges_um"] = [0, 30]
config["output"]["save_masks"] = True
config["output"]["save_qc"] = True
config["output"]["save_processing_images"] = True  # 每个文件保存全部处理图，通道颜色读取图像元数据

# 如果图像已经裁成单一脑区，使用 full_image。
# 若要分析不规则皮层/海马 ROI，可在面板选择 mask_directory。
config["tissue_roi"]["mode"] = "full_image"
config


{'input': {'image_path': '',
  'output_dir': '',
  'scene': 0,
  'time_index': 0,
  'z_projection': 'max',
  'z_index': 0,
  'zoom': 1.0,
  'pixel_size_um_x': None,
  'pixel_size_um_y': None,
  'image_width_um': None,
  'image_height_um': None},
 'channels': {'abeta': {'index': 0,
   'alias': 'Abeta',
   'gaussian_sigma_px': 1.0,
   'threshold': {'method': 'otsu',
    'value': 0.0,
    'scale': 1.0,
    'percentile': 95.0}},
  'iba1': {'index': 1,
   'alias': 'Iba1',
   'gaussian_sigma_px': 1.0,
   'threshold': {'method': 'otsu',
    'value': 0.0,
    'scale': 1.0,
    'percentile': 95.0},
   'object_filter': {'enabled': True,
    'opening_radius_px': 0,
    'closing_radius_px': 0,
    'fill_holes': False,
    'max_hole_area_um2': 0.0,
    'min_area_um2': 0.0,
    'max_area_um2': None,
    'min_circularity': 0.0,
    'min_solidity': 0.0,
    'max_eccentricity': 1.0}},
  'cd68': {'index': 2,
   'alias': 'CD68',
   'gaussian_sigma_px': 1.0,
   'threshold': {'method': 'otsu',
    'value':

## 2. 启动预览和调参面板

建议流程：添加文件 → 确认 Aβ/Iba1/CD68/DAPI 通道 → 调整强度阈值和平滑 → 调整 Iba1/CD68 对象过滤 → 在 **Microglia count** 中调整 DAPI 核形态和核周 Iba1 比例 → 调整 plaque 与距离环 → 点击 **Update preview**。通道信号按图像元数据中的原始显示色呈现；青色是保留 plaque，洋红是排除的神经元样 Aβ，DAPI 面板白色是合格细胞核、黄色是计数的小胶质细胞。选一张代表图调好后点击 **Apply current settings to all**，再逐文件检查后运行批量。每个文件的 processing_images 文件夹会保存全部处理图；最终 Excel 的 **Plaque Ring Metrics** sheet 单独保存五项 Iba1/CD68 ring 指标。生物学比较时，同一染色/成像批次应使用同一套阈值和形态规则。


**Plaque microglia gate:** cyan = retained plaque; magenta = soma-like exclusion; orange = plaque excluded because nearby DAPI+/Iba1+ microglia are below the configured minimum.


**Reuse settings:** Save session saves all selected file paths, the template, and per-file settings; Load session restores that YAML or a previous batch_parameters.yaml.


**Cumulative plaque neighborhoods:** every configured radius includes plaque pixels. For 0,15,30, outputs are plaque+0–15 µm and plaque+0–30 µm; no separate 15–30 µm region is generated.


In [3]:
brain_tuner = launch_brain_section_tuning_widget(
    config=config,
    image_paths=IMAGE_PATHS or None,
    output_root=OUTPUT_ROOT,
    max_preview_dimension=1400,
)
brain_tuner


## 3. 查看批量结果

只有在面板显示批量分析完成后再运行下面单元。统计检验应以 `animal_summary.csv` 中的每只小鼠为生物学重复，而不是把所有斑块作为独立 n。

In [ ]:
batch_result = brain_tuner.batch_result
if batch_result is None:
    print("请先点击面板中的 Run all files，并等待完成。")
else:
    batch_summary = pd.read_csv(batch_result["files"]["batch_summary"])
    image_summary = pd.read_csv(batch_result["files"]["combined_image_summary"])
    plaque_measurements = pd.read_csv(
        batch_result["files"]["combined_plaque_measurements"]
    )
    abeta_candidate_qc = pd.read_csv(
        batch_result["files"]["combined_abeta_candidate_qc"]
    )
    animal_summary = pd.read_csv(batch_result["files"]["animal_summary"])

    print(
        f"Completed: {batch_result['completed']} | "
        f"Failed: {batch_result['failed']}"
    )
    display(batch_summary)
    display(image_summary.head())
    display(plaque_measurements.head())
    display(abeta_candidate_qc.head())
    display(animal_summary)
